# Re-Ranking, Agentic RAG with LangGraph and LM Studio for Assistance in Bidding Process

In [1]:
import json 
from langchain_huggingface import HuggingFaceEmbeddings  
from langchain_chroma import Chroma  # Vector database
from langchain_community.document_loaders import DirectoryLoader, JSONLoader # Documents loader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Splitter texts in chunks

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Source data
data_directory = "source_data"

# Setting up where the vectordb directory will be stored
vectordb_directory = "rag/chroma_db"

# Variable to store the vector database
dsavectordb = None

In [3]:
def create_vectordb():
    print("\nGenerating embeddings. Wait...")
    
    # Setting the conversion schema to JSONLoader
    jq_schema = 'to_entries | map(.key + ": " + .value) | join("\\n")'
    
    # Loading the JSON files from the specific diretory
    loader = DirectoryLoader(
        data_directory,
        glob = "*.json",
        loader_cls = JSONLoader,
        loader_kwargs = {"jq_schema": jq_schema}
    )
    
    # Loading the documents from the diretory
    documents = loader.load()
    
    # Checking if there is loaded documents, otherwise, it ends the function
    if not documents:
        print("No document was found.")  
        return  
    
    # Setting a text splitter to segment the documents in small parts 
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50
    )
    
    chunks = text_splitter.split_documents(documents)
    
    # Embedding model used
    # https://huggingface.co/BAAI/bge-base-en
    model_name = "BAAI/bge-base-en"
    
    # Parametes for embeddings generation
    # Setting the embedding normalization for similarity calculation 
    encode_kwargs = {'normalize_embeddings': True}  
    
    # Model embedding instance 
    embedding_model = HuggingFaceEmbeddings(
        model_name = model_name,
        model_kwargs = {'device': 'cpu'},
        encode_kwargs = encode_kwargs
    )
    
    # Criação do banco de dados vetorial a partir dos documentos processados
    # Creating the vector database from the processed documents
    dsavectordb = Chroma.from_documents(
        chunks,
        embedding_model,
        persist_directory = vectordb_directory
    )
    
    # Message indicating that the vector database was created successfully
    print("\nRAG Vector Database Created Successfully.\n")

In [4]:
# Calling the function to generate the vector database
create_vectordb()


Generating embeddings. Wait...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5778.84it/s]



RAG Vector Database Created Successfully.

